In [31]:
import os
import sys
import argparse
import json
from tqdm.auto import tqdm
from pathlib import Path
import numpy as np
import cv2
import sys 
import supervision as sv
import ffmpeg

In [32]:
with open("/media/EVO870/datasets/prompting-mammalps-v2/metadata/label_mapping.json", "r") as f:
    label_mapping = json.load(f)

In [33]:
def from_detection_to_sv(frame_detection_results, color_by) -> sv.Detections:
    if frame_detection_results["detections"]:
        detections_list = frame_detection_results.get("detections", [])
        # Coordinates
        xyxy_coord = np.array([d["bbox"] for d in detections_list], dtype=int)

        # Core fields
        confidence = np.array([float(d["conf"]) for d in detections_list])
        tracker_id = np.array([int(d["track_id"]) for d in detections_list])

        extra_data = {}
        
        for attribute_key in ["Species", "Deer_age", "Deer_adult_sex", "Activity", "Action", "Action2"]:
            values = [d["attributes"].get(attribute_key) if "attributes" in d.keys() else None for d in detections_list] 
            extra_data[attribute_key] = np.array(values)
        
        if color_by == "action":
            class_id = np.array([int(label_mapping["actions"][detection["attributes"]["Action"]]) if "attributes" in detection.keys() else 999 for detection in frame_detection_results["detections"]])
        elif color_by == "activity":
            class_id = np.array([int(label_mapping["activities"][detection["attributes"]["Activity"]]) if "attributes" in detection.keys() else 999 for detection in frame_detection_results["detections"]])
        elif color_by == "track":
            class_id = np.array([int(detection["track_id"]) if "track_id" in detection.keys() else 999 for detection in frame_detection_results["detections"]])
        
        detections = sv.Detections(
            xyxy=xyxy_coord,
            confidence=confidence,
            class_id=class_id,
            tracker_id=tracker_id,
            data=extra_data,
        )
    else:
        detections = sv.Detections.empty()
    
    return detections

def process_video(
        source_video_path: str,
        detections_path: str,
        target_video_path: str, 
        color_by: str,
        show_attributes=True,
) -> None:

    color_lookup = sv.ColorLookup("class")

    box_annotator = sv.BoxAnnotator(color_lookup=color_lookup, thickness=2)     # BondingBox annotator instance 
    box_fill_annotator = sv.ColorAnnotator(color_lookup=color_lookup, opacity=.2)   
    label_annotator = sv.LabelAnnotator(text_scale=1.8, color_lookup=color_lookup, text_thickness=3, smart_position=False)         # Label annotator instance 
    frame_generator = sv.get_video_frames_generator(source_path=source_video_path, stride=1) # for generating frames from video
    video_info = sv.VideoInfo.from_video_path(video_path=source_video_path)
    width, height = (video_info.width, video_info.height)

    with open(detections_path, "r") as f:
        detection_results = json.load(f)["frames"]

    frame_idx = 0
    with sv.VideoSink(target_path=target_video_path, video_info=video_info, codec="mp4v") as sink:
        for frame_idx, frame in tqdm(enumerate(frame_generator)):
            if frame_idx < len(detection_results) and len(detection_results[frame_idx]["detections"])>0:
                detections = from_detection_to_sv(detection_results[frame_idx], color_by=color_by)

                #Prepare labels
                labels = []
                for i in range(len(detections.tracker_id)):
                    # Annotating labels
                    label_parts = []

                    # Add any extra fields dynamically
                    if show_attributes:
                        for key, values in detections.data.items():
                            if values is not None and len(values) > i:
                                value = values[i]
                                if isinstance(value, dict):
                                    dict_string = "\n".join(
                                        [f"{k}:{v}" for k, v in value.items() if (v is not None and v != "none")]
                                    )
                                    label_parts.append(dict_string)
                                elif (value is not None and value != "none"):
                                    label_parts.append(f"{key}: {value}")

                    labels.append("\n".join(label_parts))
                    #labels.append(" | ".join(str(detections.data[l][tracker_id]) for l in detections.data if detections.data[l][tracker_id] != "none").replace("_", " "))
                        
                # Annotating detection boxes
                annotated_frame = box_annotator.annotate(scene = frame.copy(), detections=detections) 
                annotated_frame = box_fill_annotator.annotate(scene=annotated_frame, detections=detections)
                
                annotated_label_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
            else:
                annotated_label_frame = frame.copy()

            annotated_label_frame = cv2.putText(annotated_label_frame, str(frame_idx), org=(10, 20), color=(255, 255, 255), fontFace=1, fontScale=2)   
            sink.write_frame(frame = annotated_label_frame)
    

def reencode_video_w_audio(target_video_path: str, orig_video_path: str , codec="libx264"):
    coded_video_path = str(
        Path(target_video_path).parent
        / (Path(target_video_path).stem + "_" + codec + ".mp4")
    )
    (
        ffmpeg.input(target_video_path).video
        .output(
            ffmpeg.input(orig_video_path).audio,
            coded_video_path,
            vcodec="libx264",
            pix_fmt="yuv420p",
            video_bitrate="12048k",
            acodec="copy",
            **{"profile:v": "high"},
        )
        .run(quiet=True)
    )

    os.remove(target_video_path)
    os.rename(coded_video_path, target_video_path)

In [58]:
video_path = "/media/EVO870/datasets/prompting-mammalps-v2/videos/test/S1/C6/S1_C6_F288_V0223.mp4"
annot_path = "/media/eceo_scratch_haas001/results/prompting_mammalps-v2/salma_base_vit_l_1000e_240226_mot_cl2/multi_view/S1_C6_F288_V0223.json"
output_path = "./S1_C6_F288_V0223.mp4"

In [ ]:
process_video(video_path, annot_path, output_path, color_by="track", show_attributes=True)
reencode_video_w_audio(str(output_path), video_path)

121it [00:01, 71.24it/s]